In [ ]:
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# === CONFIGURATION ===
BASE_DIR = r"F:\Android_Mobile_App\AndroidProject_3rd"
INPUT_CSV = os.path.join(BASE_DIR, "1_Project_List.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "GitHub_Android_Projects_Filtered.csv")
ENV_FILE = "All_Tokens.env"

# === LOAD TOKEN ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN_2")
if not GITHUB_TOKEN:
    raise ValueError("❌ GITHUB_TOKEN_2 not found in All_Tokens.env")

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.mercy-preview+json"  # Enables topic support
}

# === READ PROJECT NAMES FROM CSV ===
df_input = pd.read_csv(INPUT_CSV)
project_names = df_input.iloc[:, 0].dropna().astype(str).tolist()

# === FUNCTION TO FILTER FOR ANDROID PROJECTS ===
def is_android_project(repo):
    description = (repo.get("description") or "").lower()
    language = (repo.get("language") or "").lower()
    topics = repo.get("topics", [])
    return (
        "android" in description
        or "android" in topics
        or "mobile" in topics
        or language in ["java", "kotlin"]
    )

# === RATE LIMIT HANDLER ===
def handle_rate_limit(response):
    remaining = int(response.headers.get("X-RateLimit-Remaining", 1))
    reset_time = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
    if remaining < 2:
        sleep_time = max(reset_time - time.time(), 0)
        print(f"⏳ Rate limit hit. Sleeping for {int(sleep_time)} seconds...")
        time.sleep(sleep_time + 1)  # Sleep a bit extra to be safe

# === SEARCH AND SAVE PER PROJECT ===
if not os.path.exists(OUTPUT_CSV):
    pd.DataFrame(columns=[
        "searched_name", "repo_name", "owner", "full_name", "html_url",
        "description", "language", "stars", "topics", "updated_at"
    ]).to_csv(OUTPUT_CSV, index=False)

for name in project_names:
    print(f"\n🔍 Searching for: {name}")
    url = f"https://api.github.com/search/repositories?q={name}+in:name"
    response = requests.get(url, headers=headers)
    handle_rate_limit(response)

    if response.status_code != 200:
        print(f"❌ Failed for {name}: {response.status_code}")
        continue

    results = []
    for repo in response.json().get("items", []):
        if repo["name"].lower() != name.lower():
            continue  # Skip if not an exact match

        topics_url = repo["url"] + "/topics"
        topics_resp = requests.get(topics_url, headers=headers)
        handle_rate_limit(topics_resp)

        repo["topics"] = topics_resp.json().get("names", []) if topics_resp.status_code == 200 else []

        if not is_android_project(repo):
            continue

        results.append({
            "searched_name": name,
            "repo_name": repo["name"],
            "owner": repo["owner"]["login"],
            "full_name": repo["full_name"],
            "html_url": repo["html_url"],
            "description": repo.get("description", ""),
            "language": repo.get("language", ""),
            "stars": repo.get("stargazers_count", 0),
            "topics": ",".join(repo.get("topics", [])),
            "updated_at": repo.get("updated_at", "")
        })


    if results:
        pd.DataFrame(results).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
        print(f"✅ Appended results for '{name}' to {OUTPUT_CSV}")
    else:
        print(f"⚠️ No Android-related repos found for '{name}'")



🔍 Searching for: 15puzzle
✅ Appended results for '15puzzle' to F:\Android_Mobile_App\AndroidProject_3rd\GitHub_Android_Projects_Filtered.csv

🔍 Searching for: 1rramp-android
✅ Appended results for '1rramp-android' to F:\Android_Mobile_App\AndroidProject_3rd\GitHub_Android_Projects_Filtered.csv

🔍 Searching for: 2021-1-ossp2-barcode-8
✅ Appended results for '2021-1-ossp2-barcode-8' to F:\Android_Mobile_App\AndroidProject_3rd\GitHub_Android_Projects_Filtered.csv

🔍 Searching for: 2022_2_wap_app_team1
✅ Appended results for '2022_2_wap_app_team1' to F:\Android_Mobile_App\AndroidProject_3rd\GitHub_Android_Projects_Filtered.csv

🔍 Searching for: 2048-android
✅ Appended results for '2048-android' to F:\Android_Mobile_App\AndroidProject_3rd\GitHub_Android_Projects_Filtered.csv

🔍 Searching for: 2048-battles
✅ Appended results for '2048-battles' to F:\Android_Mobile_App\AndroidProject_3rd\GitHub_Android_Projects_Filtered.csv

🔍 Searching for: 4pdaclient-plus
✅ Appended results for '4pdaclient